In [0]:
client_id = dbutils.secrets.get(scope="kv-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="client-secret")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="tenant-id")

storage_account_name = "realtimestorageaccnt"

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

Define Paths

In [0]:
container_name = "data-lake"

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"

bronze_batch_online_retail_path = f"{base_path}/bronze/batch/online_retail/online_retail.csv"
silver_batch_online_retail_path = f"{base_path}/silver/batch/online_retail/"

bronze_stream_orders_path = f"{base_path}/bronze/stream/orders/"
silver_stream_orders_path = f"{base_path}/silver/stream/orders/"

Read Batch Bronze Data

In [0]:
batch_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_batch_online_retail_path)
)

display(batch_df)

In [0]:
batch_df.printSchema()

Clean Batch Data

In [0]:
from pyspark.sql.functions import col, trim, upper, to_timestamp, current_timestamp

batch_silver_df = (
    batch_df
    .dropDuplicates()
    .filter(col("InvoiceNo").isNotNull())
    .filter(col("StockCode").isNotNull())
    .filter(col("Quantity").isNotNull())
    .filter(col("UnitPrice").isNotNull())
    .withColumn("InvoiceNo", trim(col("InvoiceNo")))
    .withColumn("StockCode", trim(col("StockCode")))
    .withColumn("Description", upper(trim(col("Description"))))
    .withColumn("Country", upper(trim(col("Country"))))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn("CustomerID", col("CustomerID").cast("long"))
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate")))
    .withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))
    .withColumn("processed_at", current_timestamp())
)

Remove Invalid Business Records

In [0]:
batch_silver_df = (
    batch_silver_df
    .filter(col("Quantity") > 0)
    .filter(col("UnitPrice") > 0)
)

display(batch_silver_df)

In [0]:
display(batch_silver_df)

7. Write Batch Silver Data as Delta

In [0]:
batch_silver_df.write \
    .format("delta").mode("overwrite").save(silver_batch_online_retail_path)

8. Validate Batch Silver Data

In [0]:
silver_batch_df = spark.read.format("delta").load(silver_batch_online_retail_path)

print("Silver batch count:", silver_batch_df.count())

display(silver_batch_df)